# AND ERA5-Land Annual Test

This notebook launches a small annual ERA5-Land test for the Andrews (AND/HJA) watersheds. It uses the same extraction code as the main workflow, but filters the uploaded watershed asset to AND before launching exports.

The default run is 2001-2023 so it overlaps with the older MODIS and climate-driver columns. It exports one CSV per year to the configured Google Drive folder.


In [ ]:
REPO_URL = "https://github.com/global-river-chem/data-workflow_spatial.git"
REPO_DIR = "/content/data-workflow_spatial"

import os
import subprocess

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)


In [ ]:
!pip -q install -r requirements.txt


In [ ]:
from src.gee_spatial.auth import initialize
from src.gee_spatial.extraction import load_run_config

assets = load_run_config("config/gee-assets.yml")
products = load_run_config("config/driver-products.yml")
run_list = load_run_config("config/run-list.yml")

ee = initialize(project=assets.get("earth_engine", {}).get("project"))

print("Earth Engine project:", assets.get("earth_engine", {}).get("project"))
print("Watershed asset:", assets["watersheds"]["asset_id"])
print("Drive output folder:", assets["exports"].get("drive_folder_url"))


In [ ]:
from src.gee_spatial.extraction import load_watersheds
from src.gee_spatial.runs import choose_property_name

watersheds = load_watersheds(assets["watersheds"]["asset_id"])
property_names = watersheds.first().propertyNames().getInfo()

lter_column = choose_property_name(watersheds, "lter", alternatives=["LTER"])
site_id_column = choose_property_name(watersheds, "site_id")
lter_values = watersheds.aggregate_array(lter_column).distinct().sort().getInfo()
and_lter_value = next((value for value in lter_values if str(value).strip().lower() == "and"), None)

print("Asset rows:", watersheds.size().getInfo())
print("Asset columns:", property_names)
print("LTER column:", lter_column)
print("Site ID column:", site_id_column)
print("LTER values:", lter_values)

if and_lter_value is None:
    raise ValueError("No AND value was found in the uploaded watershed asset LTER column.")

and_watersheds = watersheds.filter(ee.Filter.eq(lter_column, and_lter_value))
selected_row_count = and_watersheds.size().getInfo()

print("AND LTER value:", and_lter_value)
print("AND watershed rows:", selected_row_count)
print("AND preview:")
print(and_watersheds.limit(5).getInfo())

if selected_row_count == 0:
    raise ValueError("No AND watershed rows were selected.")


## Launch Annual Exports

This launches one Earth Engine table export per year. If you only want to test a few years first, change `MAX_TASKS_TO_LAUNCH` to a small number like `3`.

Do not rerun this cell unless you want to launch a duplicate set of export tasks.


In [ ]:
from pathlib import Path

from src.gee_spatial.export import export_table, task_summary
from src.gee_spatial.extraction import era5_export_columns, extract_era5_land_products
from src.gee_spatial.timing import (
    datetime_label,
    task_timing_row,
    timing_rows_from_launched_tasks,
    utc_now,
    write_timing_log,
)

START_YEAR = 2001
END_YEAR = 2023
MAX_TASKS_TO_LAUNCH = None
PRODUCTS = [
    "precip",
    "temp",
    "evapotrans",
    "potential_evap",
    "snow_cover",
    "snow_water_equiv",
]

years_to_launch = list(range(START_YEAR, END_YEAR + 1))
if MAX_TASKS_TO_LAUNCH is not None:
    years_to_launch = years_to_launch[: int(MAX_TASKS_TO_LAUNCH)]

session_started_at = utc_now()
timing_log_path = Path("timing-logs") / (
    f"gee_task_timing_era5_land_AND_annual_{START_YEAR}_{END_YEAR}_"
    f"{datetime_label(session_started_at)}.csv"
)
timing_log_path.parent.mkdir(exist_ok=True)

selectors = era5_export_columns(products, product_names=PRODUCTS)
launched_tasks = []

print("Years to launch:", years_to_launch)
print("AND watershed rows:", selected_row_count)
print("Timing log:", timing_log_path)

for year in years_to_launch:
    out_name = f"era5_land_{year}_AND_watershed_extract"
    print("\nLaunching:", out_name)

    export_rows = extract_era5_land_products(
        products,
        and_watersheds,
        year=year,
        product_names=PRODUCTS,
    )

    task = export_table(
        export_rows,
        description=out_name,
        export_config=assets["exports"],
        file_name_prefix=out_name,
        selectors=selectors,
    )

    launched_at = utc_now()
    run_row = {
        "run_name": f"era5_land_AND_annual_{START_YEAR}_{END_YEAR}",
        "mode": "era5_land",
        "products": PRODUCTS,
        "year": year,
        "month": None,
        "months": None,
        "period": "annual",
        "run_group": "AND",
    }
    timing_row = task_timing_row(
        run_row=run_row,
        export_name=out_name,
        task=task,
        selected_rows=selected_row_count,
        export_destination=assets["exports"].get("destination", "drive"),
        launched_at=launched_at,
    )
    launched_tasks.append(
        {
            "name": out_name,
            "task": task,
            "launched_at": launched_at,
            "timing_row": timing_row,
        }
    )
    write_timing_log(timing_rows_from_launched_tasks(launched_tasks), timing_log_path)
    print(task_summary(task))

print("\nLaunched tasks:", len(launched_tasks))
print("Timing log:", timing_log_path)


## Poll Running Tasks

Run this cell after the launch cell. It checks the Earth Engine task status once per minute until all launched tasks finish or fail.


In [ ]:
import time

from src.gee_spatial.timing import (
    DONE_STATES,
    print_timing_summary,
    timing_rows_from_launched_tasks,
    update_task_timing_row,
    utc_now,
    write_timing_log,
)

poll_seconds = 60

if "launched_tasks" not in globals() or not launched_tasks:
    print("No tasks were launched in this session.")
else:
    while True:
        now = utc_now()
        print("\nTask status at", now.isoformat(timespec="seconds"))
        all_done = True

        for item in launched_tasks:
            item["timing_row"] = update_task_timing_row(
                item["timing_row"],
                item["task"],
                checked_at=now,
            )
            state = item["timing_row"].get("state")
            elapsed_min = float(item["timing_row"].get("elapsed_min") or 0)
            print(f"{item['name']}: {state}, elapsed {elapsed_min:.1f} min")

            if state not in DONE_STATES:
                all_done = False

        write_timing_log(timing_rows_from_launched_tasks(launched_tasks), timing_log_path)

        if all_done:
            break

        time.sleep(poll_seconds)

    print_timing_summary(timing_rows_from_launched_tasks(launched_tasks))
    print("Timing log:", timing_log_path)


## After The Exports Finish

Download the `era5_land_YYYY_AND_watershed_extract.csv` files from the configured Google Drive output folder. After they are in `Downloads`, rerun the local comparison script:

```bash
Rscript tools/plot_and_era5_modis_comparison.R
```
